# Von Neumann & CFL Lab — reconstruire la stabilité au lieu de la mémoriser

[🇫🇷 Français](../../docs/von-neumann-cfl-lab.md#fr) · [🇬🇧 English](../../docs/von-neumann-cfl-lab.md#en) · [🇪🇸 Español](../../docs/von-neumann-cfl-lab.md#es) · [🇵🇹 Português](../../docs/von-neumann-cfl-lab.md#pt)

**Règle Diderot : prédire avant d'exécuter.** Ce notebook accompagne la dérivation détaillée. Il utilise volontairement un domaine périodique pour les expériences spectrales, car c'est le cadre naturel de l'analyse de Von Neumann.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (root / 'src').exists():
        sys.path.insert(0, str(root / 'src'))
        break

from diderot_mls.von_neumann import (
    second_difference_symbol, spectral_radius_1d, spectral_radius_2d,
    phase_velocity_ratio_1d, stable_for_all_modes_1d, stable_for_all_modes_2d,
)

def d2_periodic(u):
    return np.roll(u, -1) - 2.0*u + np.roll(u, 1)

def initial_previous_periodic(u0, r, velocity_dt=None):
    # Taylor: u(-dt) = u0 - dt*v0 + 1/2*r^2*D2(u0).
    # velocity_dt means dt*v0, so no physical dx/dt is needed in this spectral toy.
    if velocity_dt is None:
        velocity_dt = np.zeros_like(u0)
    return u0 - velocity_dt + 0.5*r**2*d2_periodic(u0)

def step_periodic(u_prev, u, r):
    return 2.0*u - u_prev + r**2*d2_periodic(u)

def run_periodic(u0, r, nsteps=120):
    up = initial_previous_periodic(u0, r)
    u = u0.copy()
    maxima = [float(np.max(np.abs(u)))]
    for _ in range(nsteps):
        un = step_periodic(up, u, r)
        up, u = u, un
        maxima.append(float(np.max(np.abs(u))))
    return np.asarray(maxima)

## 1 — Une erreur minuscule peut-elle devenir gigantesque ?
On part exactement de la question de stabilité. Même bruit initial, même schéma ; seul le nombre de Courant `r` change. **Prédiction :** sous la limite, l'amplitude reste bornée ; au-dessus, une composante haute fréquence du bruit finit par dominer.

In [ ]:
rng = np.random.default_rng(20260818)
u0 = 1e-10 * rng.normal(size=256)
g_stable = run_periodic(u0, r=0.8, nsteps=180)
g_unstable = run_periodic(u0, r=1.05, nsteps=180)
plt.figure(figsize=(8,4))
plt.semilogy(g_stable, label='r = 0.80')
plt.semilogy(g_unstable, label='r = 1.05')
plt.xlabel('pas de temps'); plt.ylabel('max |erreur|')
plt.title('Une perturbation minuscule : stable vs instable')
plt.legend(); plt.grid(alpha=.25); plt.show()

## 2 — À quoi ressemble un mode de Fourier sur une grille ?
Un mode réel peut être visualisé comme `cos(j θ)`. Lorsque `θ` approche `π`, les valeurs alternent presque `+ - + -`. **Prédiction :** `θ=π` est l'oscillation la plus rapide que la grille puisse représenter.

In [ ]:
j = np.arange(32)
plt.figure(figsize=(9,4))
for theta in [0.2*np.pi, 0.6*np.pi, np.pi]:
    plt.plot(j, np.cos(j*theta), marker='o', label=f'θ/π = {theta/np.pi:.1f}')
plt.xlabel('indice de maille j'); plt.ylabel('cos(j θ)')
plt.title('Modes spatiaux de plus en plus rapides')
plt.legend(); plt.grid(alpha=.25); plt.show()

## 3 — Le stencil agit comme un opérateur à valeur propre
Pour un mode compatible avec le domaine périodique, la seconde différence doit rendre exactement la même forme multipliée par `λ(θ) = -4 sin²(θ/2)`. C'est le pont concret vers les valeurs propres.

In [ ]:
N = 128; m = 17
j = np.arange(N); theta = 2*np.pi*m/N
mode = np.exp(1j*j*theta)
lhs = d2_periodic(mode)
lam = second_difference_symbol(theta)
rhs = lam * mode
print('θ =', theta)
print('λ(θ) théorique =', lam)
print('erreur max |D2(mode) - λ mode| =', np.max(np.abs(lhs-rhs)))

## 4 — Tracer directement le facteur d'amplification
La théorie donne deux racines `G`; on trace leur module maximal `ρ(θ)`. **Prédiction :** pour `r≤1`, `ρ=1` ; pour `r>1`, les modes proches de `θ=π` passent au-dessus de 1.

In [ ]:
theta = np.linspace(0, np.pi, 500)
plt.figure(figsize=(8,4))
for r in [0.7, 1.0, 1.05]:
    plt.plot(theta/np.pi, spectral_radius_1d(r, theta), label=f'r={r}')
plt.axhline(1.0, lw=.8)
plt.xlabel('θ / π'); plt.ylabel('ρ = max |G|')
plt.title('Von Neumann : la limite CFL apparaît dans le spectre')
plt.legend(); plt.grid(alpha=.25); plt.show()
print('stable r=.7 ?', stable_for_all_modes_1d(.7))
print('stable r=1.05 ?', stable_for_all_modes_1d(1.05))

## 5 — En 2D, le domaine stable est `r_x² + r_y² ≤ 1`
On cartographie la condition dans le plan `(r_x,r_y)`. Sur une grille carrée `r_x=r_y=r`, l'intersection avec la diagonale donne `r=1/√2`.

In [ ]:
rx = np.linspace(0, 1.15, 300); ry = np.linspace(0, 1.15, 300)
RX, RY = np.meshgrid(rx, ry, indexing='xy')
stable = (RX**2 + RY**2 <= 1.0)
plt.figure(figsize=(5.5,5))
plt.imshow(stable.astype(float), origin='lower', extent=(rx[0],rx[-1],ry[0],ry[-1]), aspect='auto')
t = np.linspace(0, np.pi/2, 300)
plt.plot(np.cos(t), np.sin(t), lw=2, label='r_x² + r_y² = 1')
diag = 1/np.sqrt(2)
plt.scatter([diag],[diag], s=50, label='grille carrée : 1/√2')
plt.xlabel('r_x'); plt.ylabel('r_y'); plt.title('Région de stabilité 2D')
plt.legend(); plt.show()
print('1/sqrt(2) =', diag)
print('condition au point diagonal ?', stable_for_all_modes_2d(diag, diag))

## 6 — Stable ne signifie pas exact : dispersion numérique
On trace le rapport `v_phase,num / c`. La valeur idéale est 1. **Prédiction :** pour `r<1`, les modes courts sont davantage déformés ; le cas particulier `r=1` en 1D suit exactement la branche de phase résolue de ce schéma.

In [ ]:
theta = np.linspace(1e-4, np.pi, 500)
plt.figure(figsize=(8,4))
for r in [0.5, 0.8, 1.0]:
    plt.plot(theta/np.pi, phase_velocity_ratio_1d(r, theta), label=f'r={r}')
plt.axhline(1.0, lw=.8)
plt.xlabel('θ / π'); plt.ylabel('v_phase,num / c')
plt.title('Dispersion numérique du schéma 1D')
plt.legend(); plt.grid(alpha=.25); plt.show()

## 7 — Fermer la boucle : la théorie prédit-elle la simulation directe ?
On excite le mode le plus dangereux `θ=π`. Pour `r=1.05`, Von Neumann prédit `ρ>1`. La récurrence temporelle doit donc montrer la même croissance.

In [ ]:
N=128; j=np.arange(N); worst=(-1.0)**j
plt.figure(figsize=(8,4))
for r in [0.8, 1.05]:
    measured=run_periodic(worst, r=r, nsteps=80)
    rho=float(spectral_radius_1d(r, np.pi))
    plt.semilogy(measured, label=f'simulation r={r}, ρ={rho:.3f}')
    if rho > 1.0 + 1e-12:
        n=np.arange(len(measured))
        reference=measured[0]*rho**n
        plt.semilogy(reference, '--', label=f'ρ^n, r={r}')
plt.xlabel('pas de temps'); plt.ylabel('max |u|')
plt.title('Prédiction spectrale vs croissance directe')
plt.legend(); plt.grid(alpha=.25); plt.show()

## Fin du parcours
Nous avons reconstruit : **erreur → Fourier → mode propre discret → facteur d'amplification → spectre → CFL → dispersion numérique**.

La prochaine marche naturelle n'est plus seulement ‘est-ce stable ?’, mais ‘est-ce convergent et suffisamment précis ?’ : ordre d'erreur, étude de raffinement de maillage, dispersion/dissipation, puis comparaison avec un modèle physique plus riche comme Saint-Venant.